[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pdf-tools/components-code-sample-hub/blob/main/jupyter/pdftools_sdk/pdftools_sdk_ocr_document.ipynb)

In [ ]:
%pip install pdftools_sdk
%pip install ipython

# OCR a PDF document
Apply OCR to a PDF document to make scanned content
searchable. Text is recognized from images, existing text
is updated with correct Unicode, and tagging is added for
accessibility.

In [ ]:
import io
import io
from pdftools_sdk.pdf import Document
from pdftools_sdk.ocr import Engine, Processor, OcrOptions
from pdftools_sdk.ocr.image_processing_mode import ImageProcessingMode
from pdftools_sdk.ocr.text_processing_mode import TextProcessingMode
from pdftools_sdk.ocr.text_skip_mode import TextSkipMode
from pdftools_sdk.ocr.unicode_source import UnicodeSource
from pdftools_sdk.ocr.page_processing_mode import PageProcessingMode
from pdftools_sdk.ocr.tagging_mode import TaggingMode

In [ ]:
# Download a file from a given URL and save it to the local system
def prepare_file(url: str, path: str):
    import requests
    response = requests.get(url)
    response.raise_for_status()

    with open(path, 'wb') as f:
        f.write(response.content)

In [ ]:
# Set input arguments
ocr_engine_name = 'service@http://localhost:7982/'
language = 'German,English'
input_url = 'https://pdftools-public-downloads-production.s3.eu-west-1.amazonaws.com/samples/testfiles/InvoiceNone.pdf'
input_path = 'InvoiceNone.pdf'
prepare_file(input_url, input_path)
output_path = 'OcrOutput.pdf'

In [ ]:
def warning_handler(message: str, category, page_no: int, context: str):
    if page_no > 0:
        print(f"- {category.name}: {message} ({context} page {page_no})")
    else:
        print(f"- {category.name}: {message} ({context})")

In [ ]:
def ocr_document(ocr_engine_name: str, language: str, input_path: str, output_path: str):
    # Create the OCR engine
    with Engine.create(ocr_engine_name) as engine:

        # Set the language(s) for OCR recognition (e.g. "German,English")
        engine.languages = language

        # Open input document
        with io.FileIO(input_path, 'rb') as in_stream:
            with Document.open(in_stream) as input_document:

                # Configure OCR options
                options = OcrOptions()

                # Configure image OCR: recognize text from scanned images
                options.image_options.mode = ImageProcessingMode.UPDATE_TEXT
                options.image_options.remove_only_invisible_ocr_text = True
                options.image_options.deskew_scan = True
                options.image_options.rotate_scan = True

                # Configure text OCR: update non-extractable text with correct Unicode
                options.text_options.mode = TextProcessingMode.UPDATE
                options.text_options.skip_mode = TextSkipMode.KNOWN_SYMBOLIC
                options.text_options.unicode_source = UnicodeSource.INSTALLED_FONT

                # Configure page OCR: process all pages and add tagging for accessibility
                options.page_options.mode = PageProcessingMode.ALL
                options.page_options.tagging = TaggingMode.AUTO

                # Create the OCR processor and add a warning handler
                processor = Processor()
                processor.add_warning_handler(warning_handler)

                # Create stream for output file
                with io.FileIO(output_path, 'wb+') as output_stream:

                    # Process the document with OCR
                    processor.process(input_document, engine, output_stream, options)

In [ ]:
try:
    # By default, a test license key is active. In this case, a watermark is added to the output. 
    # If you have a license key, please uncomment the following call and set the license key.
    # from pdftools_sdk.sdk import Sdk
    # Sdk.initialize("<-- insert license key -->")

    ocr_document(ocr_engine_name, language, input_path, output_path)

    print(f"Successfully created file {output_path}")
except Exception as e:
    print(f"An error occurred: {e}")